# Nested Learning for CAD — v5.1 benchmark 7.2

Đọc `docs/main_spec/benchmark_v5_1.md` và study JSON. Method đã qua normal pilot; **AD gain chưa được xác minh**. MVTec development, không VisA. Giữ source này cho mọi phases/jobs.

Ba inputs: source `hope_cad_support_v5_1_benchmark_source_r1.zip` (Kaggle có thể giải nén), MVTec AD và locked ViT-B/8 safetensors. Không attach pilot/v4/v5 learned weights. GPU T4×2; **Save Version → Save & Run All**. Không pip upgrade để bypass runtime lock.

Chạy `preflight` trước, giữ ZIP/log. Sau đó import notebook7.1 và thêm preflight PASS output. Job mặc định NESTED_seed0. CADIC_seed0 và FIRST_ORDER_seed0 chạy separate versions để matched quick view. **7.2** cần completed7.1 của đúng JOB, chỉ thêm105cells, không train. **7.3** một registered job mỗiversion, seed1/2 và các controls; defaultsNESTED_seed1. Fullstudy27jobs, không chỉ3NESTEDseeds. Mỗi outputZIP chỉ một job +commonlocks; khi quay lại job cũ phải attach latestoutput đúngjob đó. Tháo older/preflight outputs khi đã có latestjob output, tránh ambiguity.

`SESSION_HOURS=9.0` chỉ softpause với180s export reserve, không trainingbudget/ETA. Pause: attach latest benchmark output với cùngsource/assets/job/phase. Check handoff; `paused_at_atomic_boundary` chưa xong, `*_complete_pending_review` cần review. Technicalerror phải giữfailureZIP/log và stop; không sửa learning/scoring từtestmetrics. Heartbeat30s.

Giữ ZIP `hope_cad_support_v5_1_benchmark_evidence.zip` và log tại `.agents/artifacts/runs/support_v5_1_benchmark_<JOB>/session_01/` (preflight: `support_v5_1_benchmark_preflight_01`). Resume tăngsession. Không ghi đè evidence. Không hứa ETA; full3599normal learning có thể cần nhiều sessions.


In [ ]:
PHASE='7.2'
JOB='NESTED_seed0'
SESSION_HOURS=9.0  # soft budget, không phải ETA
SOURCE_OVERRIDE=None
RESUME_OVERRIDE=None  # latest output cùng job, hoặc preflight PASS khi bắt đầu job mới
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,os,sys,time,shutil
STARTED=time.monotonic()
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_support_v5_1_benchmark_source')
print('[BOOTSTRAP] Tìm source HOPE-CAD v5.1 benchmark',flush=True)
def hash_file(p):
    with Path(p).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
def source_manifest(p):
    if p.is_dir():return json.loads((p/'bundle_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:
        if len(z.namelist())!=len(set(z.namelist())):raise ValueError('Duplicate source ZIP paths')
        m=json.loads(z.read('bundle_manifest.json'))
        if set(z.namelist())!=set(m['files'])|{'bundle_manifest.json'}:raise ValueError('Source inventory mismatch')
        return m
def target(name):
    rel=PurePosixPath(name)
    if rel.is_absolute() or '..' in rel.parts or rel.as_posix()!=name or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    p=ROOT/name
    if not p.resolve().is_relative_to(ROOT.resolve()):raise ValueError('Escaping source path')
    return p
if SOURCE_OVERRIDE:choices=[Path(SOURCE_OVERRIDE)]
else:
    choices=[]
    for p in INPUT.rglob('bundle_manifest.json'):
        if json.loads(p.read_text()).get('schema')=='hope_support_v5_1_benchmark_source_v1':choices.append(p.parent)
    if not choices:
        for p in INPUT.rglob('*.zip'):
            try:
                if source_manifest(p).get('schema')=='hope_support_v5_1_benchmark_source_v1':choices.append(p)
            except (KeyError,ValueError,zipfile.BadZipFile):pass
if len(choices)!=1:raise RuntimeError(f'Expected ONE support-v5.1 source, found {choices}; set SOURCE_OVERRIDE')
source=choices[0];manifest=source_manifest(source)
if manifest['schema']!='hope_support_v5_1_benchmark_source_v1':raise ValueError('Wrong source schema')
if ROOT.exists() and any(ROOT.iterdir()):
    previous=ROOT/'bundle_manifest.json'
    if not previous.is_file() or json.loads(previous.read_text()).get('files')!=manifest['files']:
        raise RuntimeError('Stale source directory: start a fresh kernel/saved version for a different delivery')
    existing={p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*') if p.is_file() and '__pycache__' not in p.parts}
    if existing!=set(manifest['files'])|{'bundle_manifest.json'}:
        raise RuntimeError('Unexpected files in source directory; use a fresh kernel/saved version')
ROOT.mkdir(parents=True,exist_ok=True)
if source.is_dir():
    for name,expected in manifest['files'].items():
        original=source/name;dest=target(name)
        if not original.is_file() or original.is_symlink() or hash_file(original)!=expected:raise ValueError('Source checksum mismatch: '+name)
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(original,dest)
else:
    with zipfile.ZipFile(source) as z:
        for name,expected in manifest['files'].items():
            dest=target(name);dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(name) as reader,dest.open('wb') as writer:shutil.copyfileobj(reader,writer)
            if hash_file(dest)!=expected:raise ValueError('Source checksum mismatch: '+name)
(ROOT/'bundle_manifest.json').write_text(json.dumps(manifest,sort_keys=True))
os.chdir(ROOT);sys.path.insert(0,str(ROOT));sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('[BOOTSTRAP] Verified source files:',len(manifest['files']),manifest['origin'],flush=True)
WORK=Path('/kaggle/working/hope_support_v5_1_benchmark')
ASSETS=Path('/tmp/hope_support_v5_1_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_1_benchmark_evidence.zip')


In [ ]:
from notebook_support.support_benchmark_v5_1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,job=JOB,session_hours=SESSION_HOURS,started=STARTED,inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
